# Practical 15 – ML Research: From Experiment to Paper

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University


## From one experiment to research evidence

A research paper should not report one lucky train/test split as if it were definitive evidence.

This practical uses repeated stratified cross-validation to compare several classifiers more systematically.

For every model we record:

- mean accuracy;
- standard deviation of accuracy;
- mean F1 score;
- standard deviation of F1 score.

The mean estimates typical performance, while the standard deviation gives a simple measure of variability across folds and repetitions.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

data=load_breast_cancer(as_frame=True)
X,y=data.data,data.target

models={
    "Logistic":make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000)),
    "SVM-RBF":make_pipeline(StandardScaler(),SVC(kernel="rbf")),
    "RandomForest":RandomForestClassifier(n_estimators=300,random_state=42)
}
cv=RepeatedStratifiedKFold(n_splits=5,n_repeats=3,random_state=42)

records=[]
for name,model in models.items():
    s=cross_validate(model,X,y,cv=cv,scoring=["accuracy","f1"])
    records.append({"model":name,
                    "accuracy_mean":s["test_accuracy"].mean(),
                    "accuracy_std":s["test_accuracy"].std(),
                    "f1_mean":s["test_f1"].mean(),
                    "f1_std":s["test_f1"].std()})
results=pd.DataFrame(records).sort_values("f1_mean",ascending=False)
results

## Save experiment results

Research workflows should preserve experiment outputs rather than relying on values copied manually from the notebook.

`to_csv()` writes the result table to a file that can later be used for:

- tables in a report or paper;
- plotting;
- statistical analysis;
- reproducibility checks.

A good research workflow should also record dataset version, preprocessing, random seeds, model hyperparameters, and evaluation protocol.

In [ ]:
results.to_csv("experiment_results.csv",index=False)
print("Saved experiment_results.csv")

## Research checklist
Record dataset version, preprocessing, protocol, random seeds, hyperparameters, metrics, and software versions. Do not report only the best run.